## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# Ensure sampling.yaml omits thinking_level (mapped to OpenAI reasoning_effort by LiteLLM)
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(
    'temperature: 0.2\n'
    'top_p: 0.95\n'
    'max_output_tokens: 8192\n'
    'thinking_config:\n'
    '  thinking_budget: 4096\n'
    '  include_thoughts: true\n',
    encoding='utf-8',
)

# Overlay our submission (akshay/itr-4) on the official sample submission.
import base64, json, re
OUR_FILES = json.loads(base64.b64decode('...').decode())
for rel, text in OUR_FILES.items():
    dest = AGENT_DIR / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(text, encoding='utf-8')
print(f'Overlaid {len(OUR_FILES)} files from akshay/itr-4')

# Adapter weights are not available for this run: drop adapter references.
for y in [AGENT_DIR / 'agent.yaml', *sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))]:
    y.write_text(re.sub(r'(?m)^adapter:.*\n', '', y.read_text(encoding='utf-8')), encoding='utf-8')
shutil.rmtree(AGENT_DIR / 'adapters', ignore_errors=True)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 3. Code Graph Functions

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server

In [ ]:
"""Capture exact OpenAI-compatible requests and responses sent to vLLM."""

from __future__ import annotations

import json
import threading
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import Any


class VllmTraceProxy:
    """Local transparent proxy that records chat-completion exchanges."""

    def __init__(self, upstream_base: str) -> None:
        self.upstream_base = upstream_base.rstrip('/')
        self.exchanges: list[dict[str, Any]] = []
        self._lock = threading.Lock()
        self._server: ThreadingHTTPServer | None = None
        self._thread: threading.Thread | None = None

    @property
    def api_base(self) -> str:
        assert self._server is not None
        return f'http://127.0.0.1:{self._server.server_port}/v1'

    def start(self) -> None:
        proxy = self

        class Handler(BaseHTTPRequestHandler):
            def do_POST(self) -> None:  # noqa: N802
                length = int(self.headers.get('Content-Length', '0'))
                request_body = self.rfile.read(length)
                suffix = self.path
                if proxy.upstream_base.endswith('/v1') and suffix.startswith('/v1/'):
                    suffix = suffix[3:]
                upstream_url = proxy.upstream_base + suffix
                headers = {
                    key: value
                    for key, value in self.headers.items()
                    if key.lower() not in {'host', 'content-length', 'connection'}
                }
                request = urllib.request.Request(
                    upstream_url, data=request_body, headers=headers, method='POST'
                )
                try:
                    with urllib.request.urlopen(request, timeout=900) as response:
                        status = response.status
                        response_headers = dict(response.headers.items())
                        response_body = response.read()
                except urllib.error.HTTPError as error:
                    status = error.code
                    response_headers = dict(error.headers.items())
                    response_body = error.read()
                except Exception as error:  # pragma: no cover - network failure path
                    status = 502
                    response_headers = {'Content-Type': 'application/json'}
                    response_body = json.dumps({'error': str(error)}).encode()

                if self.path.rstrip('/').endswith('/chat/completions'):
                    with proxy._lock:
                        proxy.exchanges.append({
                            'request_raw_json': request_body.decode('utf-8', errors='replace'),
                            'request': _decode_json(request_body),
                            'response_status': status,
                            'response_raw_json': response_body.decode('utf-8', errors='replace'),
                            'response': _decode_json(response_body),
                        })

                self.send_response(status)
                for key, value in response_headers.items():
                    if key.lower() not in {'content-length', 'transfer-encoding', 'connection'}:
                        self.send_header(key, value)
                self.send_header('Content-Length', str(len(response_body)))
                self.end_headers()
                self.wfile.write(response_body)

            def log_message(self, format: str, *args: object) -> None:
                return

        self._server = ThreadingHTTPServer(('127.0.0.1', 0), Handler)
        self._thread = threading.Thread(target=self._server.serve_forever, daemon=True)
        self._thread.start()

    def stop(self) -> None:
        if self._server is not None:
            self._server.shutdown()
            self._server.server_close()
        if self._thread is not None:
            self._thread.join(timeout=5)

    def write_trace(self, path: Path, *, run: dict[str, Any], final: dict[str, Any]) -> None:
        turns = []
        for index, exchange in enumerate(self.exchanges, start=1):
            response = exchange['response'] if isinstance(exchange['response'], dict) else {}
            message = ((response.get('choices') or [{}])[0].get('message') or {})
            tool_calls = _normalize_tool_calls(message.get('tool_calls', []))
            next_request = (
                self.exchanges[index]['request']
                if index < len(self.exchanges) and isinstance(self.exchanges[index]['request'], dict)
                else {}
            )
            turns.append({
                'turn': index,
                'input': {'vllm_request': exchange['request'], 'raw_json': exchange['request_raw_json']},
                'output': {
                    'vllm_response_raw': exchange['response'],
                    'raw_json': exchange['response_raw_json'],
                    'assistant_content': message.get('content'),
                    'reasoning': message.get('reasoning') or message.get('reasoning_content'),
                    'tool_calls': tool_calls,
                },
                'tool_results': _tool_results(next_request, tool_calls),
            })
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps({'schema_version': 'vllm-model-trace-v1', 'run': run, 'turns': turns, 'final': final}, indent=2) + '\n')


def _decode_json(body: bytes) -> Any:
    try:
        return json.loads(body)
    except json.JSONDecodeError:
        return None


def _normalize_tool_calls(calls: Any) -> list[dict[str, Any]]:
    """Keep raw function arguments and add parsed values/types for inspection."""
    normalized = []
    for call in calls if isinstance(calls, list) else []:
        function = call.get('function', {}) if isinstance(call, dict) else {}
        raw_args = function.get('arguments')
        parsed_args = _decode_json(raw_args.encode()) if isinstance(raw_args, str) else raw_args
        normalized.append({
            'id': call.get('id'),
            'type': call.get('type'),
            'name': function.get('name'),
            'arguments_raw_json': raw_args,
            'arguments': parsed_args,
            'argument_types': {
                key: type(value).__name__
                for key, value in parsed_args.items()
            } if isinstance(parsed_args, dict) else None,
        })
    return normalized


def _tool_results(request: dict[str, Any], calls: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Extract exact tool responses that vLLM received in the next request."""
    call_names = {call.get('id'): call.get('name') for call in calls}
    results = []
    for message in request.get('messages', []):
        if message.get('role') not in {'tool', 'tool_responses'}:
            continue
        raw_content = message.get('content')
        parsed_content = _decode_json(raw_content.encode()) if isinstance(raw_content, str) else raw_content
        call_id = message.get('tool_call_id')
        if call_id not in call_names:
            continue
        results.append({
            'tool_call_id': call_id,
            'name': call_names.get(call_id),
            'output_raw_json': raw_content,
            'output': parsed_content,
        })
    return results


In [ ]:
import litellm
import torch
from adk_submission import ModelRegistry, VllmConfig, VllmServer, discover_adapters
from google.adk.models.lite_llm import LiteLlm
from adk_submission.resolvers.generation import install_litellm_reasoning_patch
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

# Validate single base model and discover any PEFT LoRA adapters in the submission directory
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    default_chat_template_kwargs={'enable_thinking': True},
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Record the same raw model exchanges captured by the local pipeline.
trace_proxy = VllmTraceProxy(server_instance.base_url)
trace_proxy.start()

# Register the base model and discovered LoRA aliases through the recorder.
install_litellm_reasoning_patch()
target_model_id = getattr(server_instance, 'effective_model_path', vllm_cfg.model)
model_aliases = {declared_model: target_model_id, TARGET_MODEL_NAME: target_model_id}
lora_names = []
if adapters and getattr(adapters, 'adapters', None):
    lora_names.extend(adapters.adapters.keys())
if getattr(server_instance, 'lora_modules', None):
    lora_names.extend(server_instance.lora_modules.keys())
for lora_name in lora_names:
    model_aliases.setdefault(lora_name, lora_name)
    model_aliases.setdefault(f'adapter:{lora_name}', lora_name)

models = ModelRegistry()
for alias, model_name in model_aliases.items():
    if not model_name.startswith(('hosted_vllm/', 'openai/', 'custom/')):
        model_name = f'openai/{model_name}'
    models.register(alias, LiteLlm(
        model=model_name,
        api_base=trace_proxy.api_base,
        api_key=INFERENCE_API_KEY,
    ))

## 5. Run Phase 1 Inference and Phase 2 Verification

The bundled `sample_submission/` sets very low budget constraints in the `eval_config.yaml` to make testing easier. Set your own constraints below, if you like. The only enforced constraint in this competition is a 12 hour inference runtime.

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read evaluation settings from the submission's eval_config.yaml
eval_config_file = AGENT_DIR / 'eval_config.yaml'
raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

# Set your own budget constraints here
timeout_seconds = int(eval_section.get('timeout_seconds', 120))
max_tool_calls = int(eval_section.get('max_tool_calls', 30))
max_time_minutes = float(eval_section.get('max_time_minutes', 10.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

RUN_IDS = ['fastapi_14303', 'fastapi_14349', 'fastapi_14360', 'fastapi_14419', 'fastapi_14485', 'fastapi_14492', 'fastapi_14605', 'fastapi_14851', 'fastapi_5624', 'requests_7205', 'rich_3043', 'rich_3480', 'rich_3777', 'rich_4075']
by_id = {t.instance_id: t for t in tasks}
missing = [i for i in RUN_IDS if i not in by_id]
if missing:
    raise ValueError(f'Not in the competition tasks: {missing}')
SAMPLE_TASKS = [by_id[i] for i in RUN_IDS]
LOGS_ROOT = WORKING_DIR / 'logs' / 'remote' / 'train'
print(f'Running {len(SAMPLE_TASKS)} tasks; budget {max_tool_calls} calls / {max_time_minutes} min / timeout {timeout_seconds}s')
limits, gen_constraints = build_submission_limits()

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []
result = None

try:
    for idx, task in enumerate(SAMPLE_TASKS, start=1):
        print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
        task_dir = LOGS_ROOT / task.instance_id
        (task_dir / 'results').mkdir(parents=True, exist_ok=True)
        trace_proxy.exchanges.clear()
        result = None
        try:
            result = run_sync(
                evaluator.evaluate_task,
                task=task,
                task_index=idx,
                total_tasks=len(SAMPLE_TASKS),
            )
        except Exception as exc:  # keep going: one bad task must not end the run
            print(f'  -> ERROR {exc!r}')
        finally:
            final = result.model_dump(mode='json', exclude={'trace'}) if result is not None else None
            trace_proxy.write_trace(
                task_dir / 'model_trace.json',
                run={'task_id': task.instance_id, 'upstream_api_base': server_instance.base_url},
                final={'result': final},
            )
        if result is None:
            predictions.append({'id': task.instance_id, 'prediction': ''})
            continue
        patch = result.agent_patch or ''
        (task_dir / 'results' / f'{task.instance_id}.json').write_text(
            result.model_dump_json(indent=2, exclude={'trace'}), encoding='utf-8'
        )
        (task_dir / 'results' / f'{task.instance_id}.patch').write_text(patch, encoding='utf-8')
        predictions.append({'id': task.instance_id, 'prediction': patch})
        print(
            f'  -> resolved={result.resolved}, exit_code={result.test_exit_code}, '
            f'patch_chars={len(patch)}, tool_calls={result.tool_calls}, '
            f'duration={result.duration_seconds:.1f}s'
        )
finally:
    trace_proxy.stop()

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
display(submission_df)
print('NON-EMPTY PATCHES:', sum(1 for p in predictions if p['prediction']), 'of', len(predictions))


## 6. Package Submission Archive

In [ ]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')